# Credit Card Fraud Detection with Amazon SageMaker Feature Store

**Team 5** — Aligned with:
- [`Team_5_AI_Powered_Real_Time_Financial_Fraud_&_Risk_Intelligence_Platform.ipynb`](Team_5_AI_Powered_Real_Time_Financial_Fraud_&_Risk_Intelligence_Platform.ipynb) (feature engineering & ML pipeline)
- [`Sarang_Sawant_Assignment3.1.ipynb`](../../assignments/Sarang_Sawant_Assignment3.1.ipynb) (Feature Store patterns)

## Dataset

[Credit Card Fraud Detection (Kaggle)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — 284,807 European cardholder transactions (September 2013), **492 frauds (0.172%)**.

| Column | Description |
|--------|-------------|
| `Time` | Seconds since first transaction |
| `V1`–`V28` | PCA-transformed features |
| `Amount` | Transaction amount |
| `Class` | `1` = fraud, `0` = legitimate |

> **Evaluation note:** For this imbalanced dataset, use **AUPRC** (Area Under Precision-Recall Curve), not accuracy.

## Feature Groups

| Feature Group | Primary Key | Purpose |
|---------------|-------------|---------|
| `creditcard-transaction` | `transaction_id` | Per-transaction features for real-time inference |
| `hourly-fraud-risk` | `hour` | Aggregated temporal risk profile (0–23) |

**Kernel:** `Python 3 (Data Science)` on SageMaker Studio/Lab  
**IAM:** `AmazonSageMakerFullAccess`, `AmazonS3FullAccess`


## 1. Setup SageMaker Feature Store


In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import time
import os
from pathlib import Path
from time import gmtime, strftime, sleep

original_boto3_version = boto3.__version__

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

from sagemaker.session import Session
from sagemaker import get_execution_role
from sagemaker.feature_store.feature_group import FeatureGroup, AthenaQuery

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "sagemaker-featurestore-fraud"
role = get_execution_role()

print(f"Region: {region}")
print(f"S3 bucket: {default_s3_bucket_name}")
print(f"Role: {role}")



## 2. Load Credit Card Dataset

Place `creditcard.csv` from Kaggle in the notebook directory, or set `DATA_PATH` below.


In [ ]:
# Path resolution (SageMaker Lab, local repo, Colab)
NOTEBOOK_DIR = Path.cwd()
CANDIDATE_PATHS = [
    NOTEBOOK_DIR / "creditcard.csv",
    NOTEBOOK_DIR.parent / "creditcard.csv",
    Path("../../assignments/creditcard.csv"),
    Path("/content/creditcard.csv"),
]

DATA_PATH = os.environ.get("CREDITCARD_CSV_PATH")
if DATA_PATH:
    DATA_PATH = Path(DATA_PATH)
else:
    DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), CANDIDATE_PATHS[0])

print(f"Loading: {DATA_PATH}")

dtypes = {f"V{i}": "float32" for i in range(1, 29)}
dtypes.update({"Time": "float32", "Amount": "float32", "Class": "int8"})

df = pd.read_csv(DATA_PATH, dtype=dtypes)
print(f"Shape: {df.shape}")
print(f"Fraud rate: {df['Class'].mean() * 100:.4f}%")
df.head(3)



## 3. Feature Engineering

Matches Team 5 platform notebook: `Amount_Log` and `Hour`.


In [ ]:
# Assign stable transaction IDs (dataset has no native ID column)
df = df.reset_index(drop=True)
df["transaction_id"] = df.index.astype(int)

# Team 5 engineered features
df["Amount_Log"] = np.log1p(df["Amount"])
df["Hour"] = (df["Time"] // 3600) % 24

print("Class distribution:")
print(df["Class"].value_counts())
df[["transaction_id", "Time", "Amount", "Amount_Log", "Hour", "Class"]].head()



## 4. Sample for Feature Store Ingestion

Full dataset has 284K rows; we stratified-sample for lab-friendly ingestion (includes all available frauds in sample).


In [ ]:
SAMPLE_SIZE = 2000  # increase for production-scale tests

fraud_df = df[df["Class"] == 1]
legit_df = df[df["Class"] == 0]

n_fraud = min(len(fraud_df), max(50, int(SAMPLE_SIZE * 0.05)))
n_legit = SAMPLE_SIZE - n_fraud

sampled = pd.concat([
    fraud_df.sample(n=n_fraud, random_state=42),
    legit_df.sample(n=n_legit, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Sample size: {len(sampled)} (fraud={sampled['Class'].sum()}, legit={(sampled['Class']==0).sum()})")



## 5. Build Transaction Feature Group DataFrame


In [ ]:
# Rename columns for Feature Store (snake_case, no spaces)
tx_df = sampled.copy()
tx_df = tx_df.rename(columns={
    "Time": "transaction_time",
    "Amount": "amount",
    "Amount_Log": "amount_log",
    "Hour": "hour",
    "Class": "is_fraud",
})
# Lowercase PCA features: V1 -> v1
pca_rename = {f"V{i}": f"v{i}" for i in range(1, 29)}
tx_df = tx_df.rename(columns=pca_rename)

# Round for storage consistency
float_cols = tx_df.select_dtypes(include=["float32", "float64"]).columns
tx_df[float_cols] = tx_df[float_cols].round(5)

# Event time: ingestion timestamp (Python epoch seconds)
current_time_sec = int(round(time.time()))
tx_df["event_time"] = float(current_time_sec)

# Columns for feature group (exclude raw index)
tx_feature_cols = (
    ["transaction_id", "event_time", "transaction_time", "amount", "amount_log", "hour", "is_fraud"]
    + [f"v{i}" for i in range(1, 29)]
)
tx_ingest = tx_df[tx_feature_cols].copy()
tx_ingest["transaction_id"] = tx_ingest["transaction_id"].astype(int)
tx_ingest["hour"] = tx_ingest["hour"].astype(int)
tx_ingest["is_fraud"] = tx_ingest["is_fraud"].astype(int)

tx_ingest.head(3)



## 6. Build Hourly Fraud Risk Feature Group

Aggregated temporal profile — analogous to the **Neighborhood** feature group for housing (bucket-level features for inference enrichment).


In [ ]:
hourly_df = (
    sampled.groupby("Hour")
    .agg(
        transaction_count=("transaction_id", "count"),
        fraud_count=("Class", "sum"),
        avg_amount=("Amount", "mean"),
        avg_amount_log=("Amount_Log", "mean"),
        max_amount=("Amount", "max"),
    )
    .reset_index()
    .rename(columns={"Hour": "hour"})
)

hourly_df["fraud_rate"] = (hourly_df["fraud_count"] / hourly_df["transaction_count"]).round(6)
hourly_df["avg_amount"] = hourly_df["avg_amount"].round(2)
hourly_df["avg_amount_log"] = hourly_df["avg_amount_log"].round(5)
hourly_df["max_amount"] = hourly_df["max_amount"].round(2)
hourly_df["transaction_count"] = hourly_df["transaction_count"].astype(int)
hourly_df["fraud_count"] = hourly_df["fraud_count"].astype(int)
hourly_df["hour"] = hourly_df["hour"].astype(int)
hourly_df["event_time"] = float(current_time_sec)

hourly_ingest = hourly_df[
    ["hour", "event_time", "transaction_count", "fraud_count", "fraud_rate",
     "avg_amount", "avg_amount_log", "max_amount"]
]
hourly_ingest.sort_values("hour")



## 7. Create Feature Groups


In [ ]:
tx_fg_name = "creditcard-transaction-fg-" + strftime("%d-%H-%M-%S", gmtime())
hourly_fg_name = "hourly-fraud-risk-fg-" + strftime("%d-%H-%M-%S", gmtime())

tx_feature_group = FeatureGroup(name=tx_fg_name, sagemaker_session=feature_store_session)
hourly_feature_group = FeatureGroup(name=hourly_fg_name, sagemaker_session=feature_store_session)

tx_feature_group.load_feature_definitions(data_frame=tx_ingest)
hourly_feature_group.load_feature_definitions(data_frame=hourly_ingest)

print("Transaction FG:", tx_fg_name)
print("Hourly risk FG:", hourly_fg_name)



In [ ]:
def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print(f"Waiting for {feature_group.name}...")
        sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create {feature_group.name}: {status}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


tx_feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name="transaction_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)
wait_for_feature_group_creation_complete(tx_feature_group)

hourly_feature_group.create(
    s3_uri=f"s3://{default_s3_bucket_name}/{prefix}",
    record_identifier_name="hour",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)
wait_for_feature_group_creation_complete(hourly_feature_group)



In [ ]:
print("=== Transaction Feature Group DDL ===")
print(tx_feature_group.as_hive_ddl())
print("\n=== Hourly Fraud Risk Feature Group DDL ===")
print(hourly_feature_group.as_hive_ddl())



## 8. Ingest Records


In [ ]:
tx_feature_group.ingest(data_frame=tx_ingest, max_workers=3, wait=True)
print(f"Ingested {len(tx_ingest)} transaction records.")

hourly_feature_group.ingest(data_frame=hourly_ingest, max_workers=3, wait=True)
print(f"Ingested {len(hourly_ingest)} hourly risk records.")



## 9. Query Feature Values (Online Store)

**Screenshot these queries** for your assignment submission.


In [ ]:
def get_record_as_dict(feature_group_name, record_id):
    response = featurestore_runtime.get_record(
        FeatureGroupName=feature_group_name,
        RecordIdentifierValueAsString=str(record_id),
    )
    record = response.get("Record", [])
    return {f["FeatureName"]: f["ValueAsString"] for f in record} if record else None


# Pick 3 fraud + 3 legitimate transaction IDs from the ingested sample
fraud_ids = tx_ingest[tx_ingest["is_fraud"] == 1]["transaction_id"].head(3).tolist()
legit_ids = tx_ingest[tx_ingest["is_fraud"] == 0]["transaction_id"].head(3).tolist()
query_tx_ids = fraud_ids + legit_ids

print("Transaction queries:")
tx_query_rows = []
for tid in query_tx_ids:
    rec = get_record_as_dict(tx_fg_name, tid)
    label = "FRAUD" if tid in fraud_ids else "LEGIT"
    print(f"\n--- transaction_id={tid} ({label}) ---")
    if rec:
        for k, v in rec.items():
            print(f"  {k}: {v}")
        tx_query_rows.append(rec)
    else:
        print("  Record not found")

pd.DataFrame(tx_query_rows)



In [ ]:
# Query hourly risk for hours with highest fraud activity in sample
top_fraud_hours = (
    hourly_ingest.sort_values("fraud_rate", ascending=False)["hour"].head(3).tolist()
)
print("Hourly fraud-risk queries (top 3 hours by fraud_rate in sample):")
hourly_query_rows = []
for h in top_fraud_hours:
    rec = get_record_as_dict(hourly_fg_name, h)
    print(f"\n--- hour={h} ---")
    if rec:
        for k, v in rec.items():
            print(f"  {k}: {v}")
        hourly_query_rows.append(rec)
    else:
        print("  Record not found")

pd.DataFrame(hourly_query_rows)



## 10. Enrich Transaction with Hourly Risk (Inference Pattern)

Mirrors Team 5 `predict_transaction()` — lookup transaction features + hourly risk profile.


In [ ]:
def get_feature_value(record, feature_name):
    return str(list(filter(lambda r: r["FeatureName"] == feature_name, record))[0]["ValueAsString"])


def build_inference_features(transaction_id):
    """Fetch transaction + hourly risk features for model inference."""
    tx_resp = featurestore_runtime.get_record(
        FeatureGroupName=tx_fg_name,
        RecordIdentifierValueAsString=str(transaction_id),
    )
    tx_record = tx_resp.get("Record", [])
    if not tx_record:
        return None

    hour = get_feature_value(tx_record, "hour")
    hourly_resp = featurestore_runtime.get_record(
        FeatureGroupName=hourly_fg_name,
        RecordIdentifierValueAsString=hour,
    )
    hourly_record = hourly_resp.get("Record", [])

    return {
        "transaction_id": transaction_id,
        "amount": get_feature_value(tx_record, "amount"),
        "amount_log": get_feature_value(tx_record, "amount_log"),
        "hour": hour,
        "fraud_rate_hour": get_feature_value(hourly_record, "fraud_rate") if hourly_record else None,
        "avg_amount_hour": get_feature_value(hourly_record, "avg_amount") if hourly_record else None,
    }


# Demo on first fraud transaction in sample
demo_id = fraud_ids[0] if fraud_ids else tx_ingest["transaction_id"].iloc[0]
build_inference_features(demo_id)



## 11. Offline Store — Athena Queries


In [ ]:
s3_client = boto3.client("s3", region_name=region)

def wait_for_offline_store(feature_group, bucket, max_wait_min=6):
    resolved = (
        feature_group.describe()
        .get("OfflineStoreConfig", {})
        .get("S3StorageConfig", {})
        .get("ResolvedOutputS3Uri", "")
    )
    s3_prefix = resolved.replace(f"s3://{bucket}/", "")
    for _ in range(max_wait_min * 4):
        objs = s3_client.list_objects(Bucket=bucket, Prefix=s3_prefix)
        if objs.get("Contents"):
            print(f"Offline data ready: {feature_group.name}")
            return
        print(f"Waiting for offline store: {feature_group.name}...")
        sleep(15)
    print(f"Timeout waiting for {feature_group.name} — re-run Athena cell later.")

wait_for_offline_store(tx_feature_group, default_s3_bucket_name)
wait_for_offline_store(hourly_feature_group, default_s3_bucket_name)



In [ ]:
tx_query = tx_feature_group.athena_query()
fraud_id_list = ", ".join(str(i) for i in fraud_ids)

tx_sql = f"""
SELECT transaction_id, is_fraud, amount, amount_log, hour, v14, v17
FROM   "{tx_query.table_name}"
WHERE  transaction_id IN ({fraud_id_list})
  AND  NOT is_deleted
"""
print(tx_sql)
tx_query.run(
    query_string=tx_sql,
    output_location=f"s3://{default_s3_bucket_name}/{prefix}/query_results/",
)
tx_query.wait()
tx_query.as_dataframe()



In [ ]:
hourly_query = hourly_feature_group.athena_query()
hour_list = ", ".join(str(h) for h in top_fraud_hours)

hourly_sql = f"""
SELECT hour, fraud_count, fraud_rate, avg_amount, transaction_count
FROM   "{hourly_query.table_name}"
WHERE  hour IN ({hour_list})
  AND  NOT is_deleted
"""
print(hourly_sql)
hourly_query.run(
    query_string=hourly_sql,
    output_location=f"s3://{default_s3_bucket_name}/{prefix}/query_results/",
)
hourly_query.wait()
hourly_query.as_dataframe()



## 12. Build Training Dataset (Offline Join)

Join transaction features with hourly risk profile for batch model training — feeds the Team 5 Random Forest pipeline.


In [ ]:
train_sql = f"""
SELECT t.transaction_id,
       t.is_fraud,
       t.amount,
       t.amount_log,
       t.hour,
       t.v1, t.v2, t.v3, t.v4, t.v5,
       h.fraud_rate,
       h.avg_amount AS hour_avg_amount
FROM   "{tx_query.table_name}" t
LEFT JOIN "{hourly_query.table_name}" h
       ON t.hour = h.hour
WHERE  NOT t.is_deleted
"""
print(train_sql)
tx_query.run(
    query_string=train_sql,
    output_location=f"s3://{default_s3_bucket_name}/{prefix}/query_results/",
)
tx_query.wait()
training_dataset = tx_query.as_dataframe()
print(f"Training dataset shape: {training_dataset.shape}")
training_dataset.head()



## 13. Alignment with Team 5 ML Notebook

| Step | Team 5 Notebook | This Feature Store Notebook |
|------|-----------------|----------------------------|
| Load data | `creditcard.csv` | Same + path resolution |
| Engineer features | `Amount_Log`, `Hour` | Same, stored in Feature Store |
| Train model | Random Forest + SMOTE | Use Athena training dataset export |
| Real-time scoring | `predict_transaction()` | `build_inference_features()` via GetRecord |
| Evaluation | AUPRC, confusion matrix | Unchanged — use offline export |

Next: export `training_dataset` to S3 CSV and point the Team 5 XGBoost/RF training cell at it for an end-to-end MLOps flow.


## 14. Cleanup


In [ ]:
# Uncomment when finished
# tx_feature_group.delete()
# hourly_feature_group.delete()
# print("Feature groups deleted.")

